In [ ]:
import struct
import cdflib
import numpy as np
import datetime as dt
import spiceypy as spice
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import butter, filtfilt
from utilities import print_info, print_entire

def filter_out_start_config_noise(data, epoch, removal_width = 32, new_config_delay = 0.1):
    # Create a mask for the data
    noise_mask = np.zeros(len(data))

    # Mask the next samples when dt is above new_config_delay, for which a new config. has been made
    delta_t = np.diff(epoch)
    delta_t = [dt.total_seconds() for dt in delta_t]

    i = 0
    for dt in delta_t:
        if dt > new_config_delay:
            noise_mask[i:i+removal_width] = 1
        i +=1

    noise_mask[:removal_width] = 1

    data[noise_mask == 1] = np.nan

    return data

def multiply_lists_by_33matrix(list1, list2, list3, matrix):
    res1 = np.zeros(len(list1))
    res2 = np.zeros(len(list1))
    res3 = np.zeros(len(list1))
    for i in range(len(list1)):
        vector = np.array([list1[i], list2[i], list3[i]])
        res1[i] = vector[0]*matrix[0, 0] + vector[1]*matrix[0, 1] + vector[2]*matrix[0, 2]
        res2[i] = vector[0]*matrix[1, 0] + vector[1]*matrix[1, 1] + vector[2]*matrix[1, 2]
        res3[i] = vector[0]*matrix[2, 0] + vector[1]*matrix[2, 1] + vector[2]*matrix[2, 2]
    return res1, res2, res3

def multiply_lists_by_44matrix(list1, list2, list3, list4, matrix):
    res1 = np.zeros(len(list1))
    res2 = np.zeros(len(list1))
    res3 = np.zeros(len(list1))
    res4 = np.zeros(len(list1))
    for i in range(len(list1)):
        vector = np.array([list1[i], list2[i], list3[i], list4[i]])
        res1[i] = vector[0]*matrix[0, 0] + vector[1]*matrix[0, 1] + vector[2]*matrix[0, 2] + vector[3]*matrix[0, 3]
        res2[i] = vector[0]*matrix[1, 0] + vector[1]*matrix[1, 1] + vector[2]*matrix[1, 2] + vector[3]*matrix[1, 3]
        res3[i] = vector[0]*matrix[2, 0] + vector[1]*matrix[2, 1] + vector[2]*matrix[2, 2] + vector[3]*matrix[2, 3]
        res4[i] = vector[0]*matrix[3, 0] + vector[1]*matrix[3, 1] + vector[2]*matrix[3, 2] + vector[3]*matrix[3, 3]
    return res1, res2, res3, res4

def butter_highpass(cutoff, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='highpass', analog=False)
    return b, a

def highpass_filter(data, cutoff, fs, order=5):
    b, a = butter_highpass(cutoff, fs, order=order)
    valid_indices = ~np.isnan(data)
    y = filtfilt(b, a, data[valid_indices])
    return y

def convert_binstr_to_double_precision(bin_str_array):
    doubles = []
    for bin_str in bin_str_array:
        # Convert binary string to 8-byte integer
        try :
            int_val = int(bin_str, 2)
            bytes_val = int_val.to_bytes(8, byteorder='big')
            double_val = struct.unpack('>d', bytes_val)[0]
            doubles.append(double_val)
        except ValueError as e:
            doubles.append(np.nan)
    return np.array(doubles)

def extract_double_from_columns(data, col_start, col_end):
    # Slice relevant columns: shape (len_JMAG, 8)
    bytes_slice = data[:, col_start:col_end]
    # Transpose and flatten to match MATLAB logic
    bin_strs = [''.join(f'{byte:08b}' for byte in row) for row in bytes_slice]
    return convert_binstr_to_double_precision(bin_strs)

In [ ]:
# Get and rotate JMAG data for 31/03/2025

jmag_cdf = cdflib.CDF('../DATA/jmag_echoed/2025/03/31/JUICE_LU_RPWI-PPTD-LWYRPW79710_20250331T030003_V01.cdf')

jmag_epoch = jmag_cdf.varget('Epoch')
Bx = jmag_cdf.varget('LWT79713')
By = jmag_cdf.varget('LWT79714')
Bz = jmag_cdf.varget('LWT79715')

jmag_epoch = cdflib.cdfepoch.to_datetime(jmag_epoch)
jmag_epoch = np.array(jmag_epoch, dtype='datetime64[ms]').astype('O')

plt.figure(figsize=(12, 6))
plt.plot(jmag_epoch, Bx, label='Bx')
plt.plot(jmag_epoch, By, label='By')
plt.plot(jmag_epoch, Bz, label='Bz')
plt.legend(loc = 'upper right')
plt.xlabel('Date')
plt.ylabel('Magnetic Field (nT)')
plt.grid()
plt.show()

# Rotation matrix to go from JMAG frame to JUICE frame
R = np.array([
    [-7.77145961*1e-1,  8.39299198*1e-17,   -6.29320391*1e-1],
    [-9.51729314*1e-17, -1.00000000*1e0,    -1.58371803*1e-17],
    [-6.29320391*1e-1,  4.75864657*1e-17,   7.77145961*1e-1]])

# Rotate the magnetic field vectors to the JUICE frame
Bx_rot = R[0,0]*Bx + R[0,1]*By + R[0,2]*Bz
By_rot = R[1,0]*Bx + R[1,1]*By + R[1,2]*Bz
Bz_rot = R[2,0]*Bx + R[2,1]*By + R[2,2]*Bz
Bx = Bx_rot
By = By_rot
Bz = Bz_rot

plt.figure(figsize=(12, 6))
plt.plot(jmag_epoch, Bx, label='Bx')
plt.plot(jmag_epoch, By, label='By')
plt.plot(jmag_epoch, Bz, label='Bz')
plt.legend(loc = 'upper right')
plt.xlabel('Date')
plt.ylabel('Magnetic Field (nT)')
plt.grid()
plt.show()

In [ ]:
# Get and shift SW data for 31/03/2025

sw_cdf = cdflib.CDF("../DATA/wi_k0s_swe_20250331000326_20250331235905_cdaweb.cdf")
V_GSE = sw_cdf.varget('V_GSE')
WIND_pos_gse = sw_cdf.varget('SC_pos_gse')
Epoch = sw_cdf.varget('Epoch')
Epoch = cdflib.cdfepoch.to_datetime(Epoch)
Epoch = np.array([dt.datetime.strptime(str(epoch), "%Y-%m-%dT%H:%M:%S.%f000") for epoch in Epoch])

V_GSE[V_GSE < -1e5] = np.nan

# Load the necessary SPICE kernels
spice.furnsh("../SPICE/JUICE/kernels/mk/juice_ops.tm")

# Specify time (UTC)
et = spice.str2et("2025-03-31T00:00:00")

# Get Earth's position relative to the Sun in J2000 frame
pos_earth_j2000, _ = spice.spkpos("EARTH", et, "J2000", "NONE", "SUN")

distance_earth_sun = np.linalg.norm(pos_earth_j2000)

WIND_pos_sun_centered_x = WIND_pos_gse[:, 0] - distance_earth_sun

WIND_distance_sun = np.sqrt(WIND_pos_sun_centered_x**2 + WIND_pos_gse[:, 1]**2 + WIND_pos_gse[:, 2]**2)

# Calculate the average of SC_distance_sun
avrg_WIND_distance_sun = np.nanmean(WIND_distance_sun)

pos_juice_j2000, _ = spice.spkpos("JUICE", et, "J2000", "NONE", "SUN")
distance_juice_sun = np.linalg.norm(pos_juice_j2000)

print(f"Distance from JUICE to Sun: {distance_juice_sun} km")
print(f"Distance from WIND to Sun: {avrg_WIND_distance_sun} km")

delta_distance = avrg_WIND_distance_sun - distance_juice_sun
sw_speed = -np.nanmean(V_GSE[:, 0], axis=0)
delta_time = delta_distance / sw_speed
print(f"Delta time: {delta_time/(60*60)} hours")

# Plot before shifting
plt.figure(figsize=(12, 6))
plt.plot(Epoch, V_GSE[:, 0], label='Shifted Vx')
plt.title('Unshifted Solar Wind Velocity Component Vx Over Time')
plt.xlabel('Epoch')
plt.ylabel('Velocity (km/s)')
plt.legend()
plt.grid()
plt.show()

# Shift the solar wind data to delta_time seconds earlier this way the solar wind data is plotted on the time it reaches JUICE
SW_epoch = Epoch - dt.timedelta(seconds=delta_time)
plt.figure(figsize=(12, 6))
plt.plot(SW_epoch, V_GSE[:, 0], label='Shifted Vx')
plt.title('Shifted Solar Wind Velocity Component Vx Over Time')
plt.xlabel('Epoch')
plt.ylabel('Velocity (km/s)')
plt.legend()
plt.grid()
plt.show()

spice.unload("SPICE/JUICE/kernels/mk/juice_ops.tm")

In [ ]:
# Calculate v cross B for 31/03/2025

# X axis for JUICE points away from HGA antenna, so AWAY from the Sun, so SW speed needs to be positive

cross_products = np.zeros((len(jmag_epoch), 3))

for i in range(len(jmag_epoch)):
    # Calculate the absolute time difference
    time_diffs = np.abs(SW_epoch - jmag_epoch[i])
    # Sort the indices of time differences in ascending order
    sorted_indices = np.argsort(time_diffs)
    
    # Find the first non-nan Vx value
    sw_vx = np.nan
    for idx in sorted_indices:
        if not np.isnan(V_GSE[idx, 0]):
            sw_vx = V_GSE[idx, 0]
            break
    sw_vx = -sw_vx  # In JUICE frame, the velocity is negative
    
    cross_products[i] = [0, -float(-Bz[i] * sw_vx * 1e-6), -float(By[i] * sw_vx * 1e-6)]

EyvCB = cross_products[:, 1]
EzvCB = cross_products[:, 2]

# Define the start and end times for the desired period
start_time = dt.datetime(2025, 3, 31, 2, 50, 0)
end_time = dt.datetime(2025, 3, 31, 3, 25, 0)

# Filter the solar wind data for the specified time range
filtered_indices = [i for i, t in enumerate(SW_epoch) if start_time <= t <= end_time]
SW_epoch_filtered = SW_epoch[filtered_indices]
V_GSE_filtered = V_GSE[filtered_indices, 0]

# Plot all of the solar wind data
plt.figure(figsize=(12, 6))
plt.plot(SW_epoch, V_GSE[:, 0], label='Vx')
plt.title('Solar Wind Velocity Component Vx Over Time')
plt.xlabel('Epoch')
plt.ylabel('SW Velocity (km/s)')
plt.legend()
plt.grid()
plt.show()

# Plot the filtered solar wind data
plt.figure(figsize=(12, 6))
plt.plot(SW_epoch_filtered, V_GSE_filtered, label='Filtered Vx')
plt.title('Filtered Solar Wind Velocity Component Vx Over Time')
plt.xlabel('Epoch')
plt.ylabel('SW Velocity (km/s)')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(jmag_epoch, By, label='By')
plt.plot(jmag_epoch, Bz, label='Bz')
plt.xlabel('Epoch')
plt.ylabel('Magnetic Field (nT)')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(jmag_epoch, EyvCB*1e3, label='Ey (-v x B)')
plt.plot(jmag_epoch, EzvCB*1e3, label='Ez (-v x B)')
plt.title('Cross Products: Ey and Ez Over Time')
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Get E field for 31/03/2025

nb_final_removed = 11000

LP_cdf = cdflib.CDF('../DATA/JUICE_L1a_RPWI-LP-SID1_RICH_DE763_SNAP_20250331T030011_V02.cdf')

lp_epoch = LP_cdf.varget('Epoch')[:-nb_final_removed]
TM1 = LP_cdf.varget('LP_DATA')[:, 0][:-nb_final_removed]
TM2 = LP_cdf.varget('LP_DATA')[:, 1][:-nb_final_removed]
TM3 = LP_cdf.varget('LP_DATA')[:, 2][:-nb_final_removed]
TM4 = LP_cdf.varget('LP_DATA')[:, 3][:-nb_final_removed]
lp_epoch = cdflib.cdfepoch.to_datetime(lp_epoch)
lp_epoch = np.array(lp_epoch, dtype='datetime64[ms]').astype('O')

delta2volt = np.array([
    [-1.0, -1.0, -1.0, 1.0],
    [0.0, -1.0, -1.0, 1.0],
    [0.0, 0.0, -1.0, 1.0],
    [0.0, 0.0, 0.0, 1.0]])

volt2delta = np.array([
    [-1.0, 1.0, 0.0, 0.0],
    [0.0, -1.0, 1.0, 0.0],
    [0.0, 0.0, -1.0, 1.0],
    [0.0, 0.0, 0.0, 1.0]])

volt2E = np.array([
    [0.1852, 0.1923, 0.1917],
    [0.1320, -0.0112, -0.0322],
    [0.0, 0.1398, 0.0]])

TM2delta = np.array([[5.15*1e-6], [4.97*1e-6], [5.07*1e-6], [9.94*1e-5]])

# Define the start and end dates for filtering
start_date = dt.datetime(2025, 3, 31, 3, 0, 0)
end_date = dt.datetime(2025, 3, 31, 3, 15, 0)

# Filter the TM lists and epoch list
filtered_indices = [i for i, t in enumerate(lp_epoch) if start_date <= t <= end_date]
#lp_epoch = lp_epoch[filtered_indices]
#TM1 = TM1[filtered_indices]
#TM2 = TM2[filtered_indices]
#TM3 = TM3[filtered_indices]
#TM4 = TM4[filtered_indices]

plt.figure(figsize=(12, 6))
plt.plot(lp_epoch, TM1, label='TM1')
plt.plot(lp_epoch, TM2, label='TM2')
plt.plot(lp_epoch, TM3, label='TM3')
plt.plot(lp_epoch, TM4, label='TM4')
plt.xlabel('Epoch')
plt.ylabel('TM units')
plt.legend()
plt.grid()
plt.show()

U12 = TM1 * TM2delta[0, 0]
U23 = TM2 * TM2delta[1, 0]
U34 = TM3 * TM2delta[2, 0]
U40 = TM4 * TM2delta[3, 0]

U12 = filter_out_start_config_noise(U12, lp_epoch)
U23 = filter_out_start_config_noise(U23, lp_epoch)
U34 = filter_out_start_config_noise(U34, lp_epoch)
U40 = filter_out_start_config_noise(U40, lp_epoch)

plt.figure(figsize=(12, 6)) 
plt.plot(lp_epoch, U12, label='U12 (uncalibrated)')
plt.plot(lp_epoch, U23, label='U23 (uncalibrated)')
plt.plot(lp_epoch, U34, label='U34 (uncalibrated)')
plt.plot(lp_epoch, U40, label='U40 (uncalibrated)')
plt.xlabel('Epoch')
plt.ylabel('Voltage difference (V)')
plt.legend()
plt.grid()
plt.show()

U1, U2, U3, U4 = multiply_lists_by_44matrix(U12, U23, U34, U40, delta2volt)

plt.figure(figsize=(12, 6))
plt.plot(lp_epoch, U1, label='U1 (uncalibrated)')
plt.plot(lp_epoch, U2, label='U2 (uncalibrated)')
plt.plot(lp_epoch, U3, label='U3 (uncalibrated)')
plt.plot(lp_epoch, U4, label='U4 (uncalibrated)')
plt.xlabel('Epoch')
plt.ylabel('Voltage (V)')
plt.legend()
plt.grid()
plt.show()

"""
mean_diff_U1 = np.nanmean(np.array(U4) - np.array(U1))
mean_diff_U2 = np.nanmean(np.array(U4) - np.array(U2))
mean_diff_U3 = np.nanmean(np.array(U4) - np.array(U3))

U1 = np.array([float(u) + float(mean_diff_U1) for u in U1])
U2 = np.array([float(u) + float(mean_diff_U2) for u in U2])
U3 = np.array([float(u) + float(mean_diff_U3) for u in U3])
"""

plt.figure(figsize=(12, 6))
plt.plot(lp_epoch, U1, label='U1 (calibrated)')
plt.plot(lp_epoch, U2, label='U2 (calibrated)')
plt.plot(lp_epoch, U3, label='U3 (calibrated)')
plt.plot(lp_epoch, U4, label='U4 (calibrated)')
plt.xlabel('Epoch')
plt.ylabel('Voltage (V)')
plt.legend()
plt.grid()
plt.show()

U12, U23, U34, U40 = multiply_lists_by_44matrix(U1, U2, U3, U4, volt2delta)

plt.figure(figsize=(12, 6))
plt.plot(lp_epoch, U12, label='U12 (calibrated)')
plt.plot(lp_epoch, U23, label='U23 (calibrated)')
plt.plot(lp_epoch, U34, label='U34 (calibrated)')
plt.plot(lp_epoch, U40, label='U40 (calibrated)')
plt.xlabel('Epoch')
plt.ylabel('Voltage difference (V)')
plt.legend()
plt.grid()
plt.show()

Ex, Ey, Ez = multiply_lists_by_33matrix(U12, U23, U34, volt2E)

plt.figure(figsize=(12, 6))
plt.plot(lp_epoch, Ex*1e3, label='Ex (calibrated)')
plt.plot(lp_epoch, Ey*1e3, label='Ey (calibrated)')
plt.plot(lp_epoch, Ez*1e3, label='Ez (calibrated)')
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.legend()
plt.grid()
plt.show()

nb_points_avg = 1000

# Smooth E-field data using a 10-point rolling average
Ex_smooth = np.convolve(Ex, np.ones(nb_points_avg)/nb_points_avg, mode='valid')
Ey_smooth = np.convolve(Ey, np.ones(nb_points_avg)/nb_points_avg, mode='valid')
Ez_smooth = np.convolve(Ez, np.ones(nb_points_avg)/nb_points_avg, mode='valid')

# Adjust the epoch to match the smoothed data length
lp_epoch_smooth = lp_epoch[:len(Ex_smooth)]

plt.figure(figsize=(12, 6))
#plt.plot(lp_epoch_smooth, Ex_smooth*1e3, label='Ex (smoothed)')
plt.plot(lp_epoch_smooth, Ey_smooth*1e3, label='Ey (LP data)')
plt.plot(lp_epoch_smooth, Ez_smooth*1e3, label='Ez (LP data)')
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Trying a high-pass filter on the v cross B data

cutoff = 0.2
sampling_rate = 1 / (jmag_epoch[1] - jmag_epoch[0]).total_seconds()
order = 5

# Filter the Ey and Ez data
EyvCB_filtered = highpass_filter(EyvCB, cutoff, sampling_rate, order)
EzvCB_filtered = highpass_filter(EzvCB, cutoff, sampling_rate, order)

# Plot the filtered Ey and Ez data
plt.figure(figsize=(12, 6))
plt.scatter(jmag_epoch, EyvCB_filtered*1e3, label='Filtered Ey (-v x B)', color = 'blue', s=1)
plt.scatter(jmag_epoch, EzvCB_filtered*1e3, label='Filtered Ez (-v x B)', color = 'orange', s=1)
plt.title('Filtered Cross Products: Ey and Ez Over Time')
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Trying a high-pass filter on the LP E-field data

# Define the cutoff frequency and sampling frequency
cutoff = 0.2  # Desired cutoff frequency of the filter, Hz
sampling_rate = 1 / (lp_epoch[1] - lp_epoch[0]).total_seconds()  # Sampling frequency, Hz
order = 5

# Separate the data into different sections based on time gaps larger than 1 second
sections = []
current_section = [0]  # Start with the first index

for i in range(1, len(lp_epoch)):
    time_diff = (lp_epoch[i] - lp_epoch[i - 1]).total_seconds()
    if time_diff > 0.1:
        sections.append(current_section)
        current_section = [i]
    else:
        current_section.append(i)

if current_section:
    sections.append(current_section)

# Create separate lists for Ex, Ey, Ez based on the sections
Ex_sections = [Ex[section] for section in sections]
Ey_sections = [Ey[section] for section in sections]
Ez_sections = [Ez[section] for section in sections]
lp_epoch_sections = [lp_epoch[section] for section in sections]

# Apply the high-pass filter to each section individually and then concatenate the results
Ex_filtered_sections = []
Ey_filtered_sections = []
Ez_filtered_sections = []
lp_epoch_filtered_sections = []

for Ex_section, Ey_section, Ez_section, lp_epoch_section in zip(Ex_sections, Ey_sections, Ez_sections, lp_epoch_sections):
    valid_indices = ~np.isnan(Ex_section) & ~np.isnan(Ey_section) & ~np.isnan(Ez_section)
    Ex_filtered = highpass_filter(Ex_section[valid_indices], cutoff, sampling_rate, order)
    Ey_filtered = highpass_filter(Ey_section[valid_indices], cutoff, sampling_rate, order)
    Ez_filtered = highpass_filter(Ez_section[valid_indices], cutoff, sampling_rate, order)
    lp_epoch_filtered = lp_epoch_section[valid_indices]
    
    Ex_filtered_sections.append(Ex_filtered)
    Ey_filtered_sections.append(Ey_filtered)
    Ez_filtered_sections.append(Ez_filtered)
    lp_epoch_filtered_sections.append(lp_epoch_filtered)

# Concatenate the filtered sections back into a single array
Ex_filtered_full = np.concatenate(Ex_filtered_sections)
Ey_filtered_full = np.concatenate(Ey_filtered_sections)
Ez_filtered_full = np.concatenate(Ez_filtered_sections)
lp_epoch_filtered_full = np.concatenate(lp_epoch_filtered_sections)

plt.figure(figsize=(12, 6))
plt.scatter(lp_epoch_filtered_full, Ex_filtered_full*1e3, label='Ex (filtered)', s=1, color = 'blue')
plt.scatter(lp_epoch_filtered_full, Ey_filtered_full*1e3, label='Ey (filtered)', s=1, color = 'orange')
plt.scatter(lp_epoch_filtered_full, Ez_filtered_full*1e3, label='Ez (filtered)', s=1, color = 'green')
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Plot filtered E-field data

plt.figure(figsize=(12, 6))

plt.subplot(2, 2, 1)
plt.plot(lp_epoch_filtered_full, Ey_filtered_full*1e3, label='Ey (LP)')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.grid()

plt.subplot(2, 2, 2)
plt.plot(jmag_epoch, EyvCB_filtered*1e3, label='Ey (v x B)')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.grid()

plt.subplot(2, 2, 3)
plt.plot(lp_epoch_filtered_full, Ez_filtered_full*1e3, label='Ez (LP)')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.grid()

plt.subplot(2, 2, 4)
plt.plot(jmag_epoch, EzvCB_filtered*1e3, label='Ez (v x B)')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Electric Field (mV/m)')
plt.grid()

plt.show()

In [ ]:
# Reading jmag_echoed data from 21/08/2024

jmag_cdf = cdflib.CDF('../DATA/jmag_echoed/2024/08/21/JUICE_LU_RPWI-PPTD-LWYRPW79700_20240821T042902_V01.cdf')

NMD = jmag_cdf.varget('NMD')
epoch = jmag_cdf.varget('Epoch')
epoch = cdflib.cdfepoch.to_datetime(epoch)
epoch = np.array(epoch, dtype='datetime64[ms]').astype('O')

# Find the index where the separation occurs
epoch_diffs = np.diff(epoch)
separation_index = np.argmax(epoch_diffs > dt.timedelta(hours=1)) + 1

epoch1 = epoch[:separation_index]
epoch2 = epoch[separation_index:]

JMAG_Bx = extract_double_from_columns(NMD, 9, 17)
JMAG_By = extract_double_from_columns(NMD, 17, 25)
JMAG_Bz = extract_double_from_columns(NMD, 25, 33)

JMAG_Bx = np.where(np.abs(JMAG_Bx) < 1e-9, np.nan, JMAG_Bx)
JMAG_By = np.where(np.abs(JMAG_By) < 1e-9, np.nan, JMAG_By)
JMAG_Bz = np.where(np.abs(JMAG_Bz) < 1e-9, np.nan, JMAG_Bz)

JMAG_Bx1 = JMAG_Bx[:separation_index]
JMAG_By1 = JMAG_By[:separation_index]
JMAG_Bz1 = JMAG_Bz[:separation_index]

JMAG_Bx2 = JMAG_Bx[separation_index:]
JMAG_By2 = JMAG_By[separation_index:]
JMAG_Bz2 = JMAG_Bz[separation_index:]

plt.figure(figsize=(12, 12))

plt.subplot(3, 2, 1)
plt.scatter(epoch1, JMAG_Bx1, label='Bx', s = 1)
plt.xlabel('Epoch')
plt.ylabel('Bx (nT)')
plt.grid()
plt.legend()

plt.subplot(3, 2, 2)
plt.scatter(epoch2, JMAG_Bx2, label='Bx', s = 1)
plt.xlabel('Epoch')
plt.ylabel('Bx (nT)')
plt.grid()
plt.legend()

plt.subplot(3, 2, 3)
plt.scatter(epoch1, JMAG_By1, label='By', s = 1)
plt.xlabel('Epoch')
plt.ylabel('By (nT)')
plt.grid()
plt.legend()

plt.subplot(3, 2, 4)
plt.scatter(epoch2, JMAG_By2, label='By', s = 1)
plt.xlabel('Epoch')
plt.ylabel('By (nT)')
plt.grid()
plt.legend()

plt.subplot(3, 2, 5)
plt.scatter(epoch1, JMAG_Bz1, label='Bz', s = 1)
plt.xlabel('Epoch')
plt.ylabel('Bz (nT)')
plt.grid()
plt.legend()

plt.subplot(3, 2, 6)
plt.scatter(epoch2, JMAG_Bz2, label='Bz', s = 1)
plt.xlabel('Epoch')
plt.ylabel('Bz (nT)')
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Get and shift SW data from 2024/08/21

sw_cdf = cdflib.CDF('../DATA/wi_k0s_swe_20240820120039_20240821225846_cdaweb.cdf')

V_GSE = sw_cdf.varget('V_GSE')
WIND_pos_gse = sw_cdf.varget('SC_pos_gse')
Epoch = sw_cdf.varget('Epoch')
Epoch = cdflib.cdfepoch.to_datetime(Epoch)
Epoch = np.array([dt.datetime.strptime(str(epoch), "%Y-%m-%dT%H:%M:%S.%f000") for epoch in Epoch])

V_GSE[V_GSE < -1e5] = np.nan

# Load the necessary SPICE kernels
spice.furnsh("../SPICE/JUICE/kernels/mk/juice_ops.tm")

# Specify time (UTC)
et = spice.str2et("2024-08-21T12:00:00")

# Get Earth's position relative to the Sun in J2000 frame
pos_earth_j2000, _ = spice.spkpos("EARTH", et, "J2000", "NONE", "SUN")

distance_earth_sun = np.linalg.norm(pos_earth_j2000)

WIND_pos_sun_centered_x = WIND_pos_gse[:, 0] - distance_earth_sun

WIND_distance_sun = np.sqrt(WIND_pos_sun_centered_x**2 + WIND_pos_gse[:, 1]**2 + WIND_pos_gse[:, 2]**2)

# Calculate the average of SC_distance_sun
avrg_WIND_distance_sun = np.nanmean(WIND_distance_sun)

pos_juice_j2000, _ = spice.spkpos("JUICE", et, "J2000", "NONE", "SUN")
distance_juice_sun = np.linalg.norm(pos_juice_j2000)

print(f"Distance from JUICE to Sun: {distance_juice_sun} km")
print(f"Distance from WIND to Sun: {avrg_WIND_distance_sun} km")

delta_distance = avrg_WIND_distance_sun - distance_juice_sun
sw_speed = -np.nanmean(V_GSE[:, 0], axis=0)
delta_time = delta_distance / sw_speed
print(f"Delta time: {delta_time/(60*60)} hours")

# Plot before shifting
plt.figure(figsize=(12, 6))
plt.plot(Epoch, V_GSE[:, 0], label='Shifted Vx')
plt.title('Unshifted Solar Wind Velocity Component Vx Over Time')
plt.xlabel('Epoch')
plt.ylabel('Velocity (km/s)')
plt.legend()
plt.grid()
plt.show()

# Shift the solar wind data to delta_time seconds earlier this way the solar wind data is plotted on the time it reaches JUICE
SW_epoch = Epoch - dt.timedelta(seconds=delta_time)
plt.figure(figsize=(12, 6))
plt.plot(SW_epoch, V_GSE[:, 0], label='Shifted Vx')
plt.title('Shifted Solar Wind Velocity Component Vx Over Time')
plt.xlabel('Epoch')
plt.ylabel('Velocity (km/s)')
plt.legend()
plt.grid()
plt.show()

spice.unload("SPICE/JUICE/kernels/mk/juice_ops.tm")

In [ ]:
# Get E-f data from 2024/08/21

ef_cdf = cdflib.CDF('../DATA/JUICE_L1a_RPWI-LP-SID1_RICH_DE763_SNAP_20240821T161043_V03.cdf')

epoch = ef_cdf.varget('Epoch')
epoch = cdflib.cdfepoch.to_datetime(epoch)
epoch = np.array(epoch, dtype='datetime64[ms]').astype('O')

E_f = ef_cdf.varget('LP_DATA_ENG')
mux = ef_cdf.varget('ADC_MUX')

U12 = filter_out_start_config_noise(E_f[:, 0], epoch)
U23 = filter_out_start_config_noise(E_f[:, 1], epoch)
U34 = filter_out_start_config_noise(E_f[:, 2], epoch)
U40 = filter_out_start_config_noise(E_f[:, 3], epoch)

volt2E = np.array([
    [0.1852, 0.1923, 0.1917],
    [0.1320, -0.0112, -0.0322],
    [0.0, 0.1398, 0.0]])

delta2volt = np.array([
    [-1.0, -1.0, -1.0, 1.0],
    [0.0, -1.0, -1.0, 1.0],
    [0.0, 0.0, -1.0, 1.0],
    [0.0, 0.0, 0.0, 1.0]])

volt2delta = np.array([
    [-1.0, 1.0, 0.0, 0.0],
    [0.0, -1.0, 1.0, 0.0],
    [0.0, 0.0, -1.0, 1.0],
    [0.0, 0.0, 0.0, 1.0]])

U1, U2, U3, U4 = multiply_lists_by_44matrix(U12, U23, U34, U40, delta2volt)

mean_diff_U1 = np.nanmean(np.array(U4) - np.array(U1))
mean_diff_U2 = np.nanmean(np.array(U4) - np.array(U2))
mean_diff_U3 = np.nanmean(np.array(U4) - np.array(U3))

U1 = np.array([float(u) + float(mean_diff_U1) for u in U1])
U2 = np.array([float(u) + float(mean_diff_U2) for u in U2])
U3 = np.array([float(u) + float(mean_diff_U3) for u in U3])

U12, U23, U34, U40 = multiply_lists_by_44matrix(U1, U2, U3, U4, volt2delta)

Ex, Ey, Ez = multiply_lists_by_33matrix(U1, U2, U3, volt2E)

plt.figure(figsize=(12, 6))
#plt.plot(epoch, Ex*1e3, label='Ex')
plt.plot(epoch, Ey*1e3, label='Ey')
plt.plot(epoch, Ez*1e3, label='Ez')
plt.xlabel('Epoch')
plt.ylabel('Electric Field (V/m)')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Reading jmag_echoed data from 02/07/2024

jmag_cdf = cdflib.CDF('../DATA/jmag_echoed/2024/07/02/JUICE_LU_RPWI-PPTD-LWYRPW79700_20240702T003500_V02.cdf')
NMD = jmag_cdf.varget('NMD')
epoch = jmag_cdf.varget('Epoch')
epoch = cdflib.cdfepoch.to_datetime(epoch)
epoch = np.array(epoch, dtype='datetime64[ms]').astype('O')

JMAG_Bx = extract_double_from_columns(NMD, 9, 17)
JMAG_By = extract_double_from_columns(NMD, 17, 25)
JMAG_Bz = extract_double_from_columns(NMD, 25, 33)

JMAG_Bx = np.where(np.abs(JMAG_Bx) < 1e-9, np.nan, JMAG_Bx)
JMAG_By = np.where(np.abs(JMAG_By) < 1e-9, np.nan, JMAG_By)
JMAG_Bz = np.where(np.abs(JMAG_Bz) < 1e-9, np.nan, JMAG_Bz)

plt.figure(figsize=(12, 12))

plt.subplot(3, 1, 1)
plt.scatter(epoch, JMAG_Bx, label='Bx', s = 1)
plt.xlabel('Epoch')
plt.ylabel('Bx (nT)')
plt.grid()
plt.legend()

plt.subplot(3, 1, 2)
plt.scatter(epoch, JMAG_By, label='By', s = 1)
plt.xlabel('Epoch')
plt.ylabel('By (nT)')
plt.grid()
plt.legend()

plt.subplot(3, 1, 3)
plt.scatter(epoch, JMAG_Bz, label='Bz', s = 1)
plt.xlabel('Epoch')
plt.ylabel('Bz (nT)')
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()